# Multimodal Readmission Prediction

This notebook follows one central question: which modality is most useful for predicting 30-day hospital readmission?

I start from three different data sources:
- time-series EHR features
- clinical notes
- image features

The workflow is intentionally staged:
1. Build a baseline for each modality.
2. Compare their standalone performance.
3. Fuse the learned representations with an MLP to capture nonlinear interactions.
4. Test whether ClinicalBERT improves note representation under the 512-token constraint.
5. Conclude with the submission choice that performed best in the Kaggle competition.

My initial hypothesis was that notes would be the strongest single modality because they contain the richest clinical narrative, but I also expected the EHR and image features to add complementary signal once the model could combine them nonlinearly.

In [1]:
import os
import glob
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd

SEED = 2025
np.random.seed(SEED)

DATA_DIR = "/Users/alex/30-days-hospital-readmission-rate/2025-fall-stat-3612-group-project"
EHR_DATA_DIR = DATA_DIR
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VAL_PATH = os.path.join(DATA_DIR, "valid.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
NOTES_PATH = os.path.join(DATA_DIR, "notes.csv")
EHR_PKL_PATH = os.path.join(DATA_DIR, "ehr_preprocessed_seq_by_day_cat_embedding.pkl")
IMAGE_DIR = os.path.join(DATA_DIR, "image_features", "cxr_features")

print("Loading core datasets...")
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
notes_df = pd.read_csv(NOTES_PATH)
notes_df["text"] = notes_df["text"].fillna("").astype(str)

print(f"train_df shape: {train_df.shape}")
print(f"val_df shape:   {val_df.shape}")
print(f"test_df shape:  {test_df.shape}")
print(f"notes_df shape: {notes_df.shape}")

modified_train_unique = (
    train_df[["id", "readmitted_within_30days"]]
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)
modified_valid_unique = (
    val_df[["id", "readmitted_within_30days"]]
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)
modified_test_unique = (
    test_df[["id"]]
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)

print("\nUnique admissions (by id):")
print(f"Train: {modified_train_unique.shape}")
print(f"Valid: {modified_valid_unique.shape}")
print(f"Test:  {modified_test_unique.shape}")

train_ids = set(modified_train_unique["id"])
val_ids = set(modified_valid_unique["id"])
test_ids = set(modified_test_unique["id"])

notes_grouped = (
    notes_df.groupby("id", as_index=False)["text"]
    .agg(" ".join)
)
print(f"\nNotes grouped shape: {notes_grouped.shape}")

print("Loading EHR pickle...")
ehr_data = pd.read_pickle(EHR_PKL_PATH)
feat_dict = ehr_data["feat_dict"]
example_id_ehr = next(iter(feat_dict.keys()))
n_ehr_features = feat_dict[example_id_ehr].shape[1]
print(f"EHR feature dimension: {n_ehr_features}")

print("Loading CXR image features...")
all_df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
img_map = all_df[["id", "image_path"]].copy()
all_pkl_files = glob.glob(os.path.join(IMAGE_DIR, "*.pkl"))
print(f"Found {len(all_pkl_files)} image feature files")

pkl_index = {}
for f in all_pkl_files:
    base = os.path.basename(f)
    stem = base.replace(".pkl", "")
    pkl_index[stem] = f

image_temp_train = defaultdict(list)
image_temp_val = defaultdict(list)
image_temp_test = defaultdict(list)

for _, row in img_map.iterrows():
    adm_id = row["id"]
    img_path = row["image_path"]
    if img_path not in pkl_index:
        continue
    with open(pkl_index[img_path], "rb") as f:
        feat = pickle.load(f)
    feat = np.asarray(feat, dtype=np.float32)

    if adm_id in train_ids:
        image_temp_train[adm_id].append(feat)
    elif adm_id in val_ids:
        image_temp_val[adm_id].append(feat)
    elif adm_id in test_ids:
        image_temp_test[adm_id].append(feat)

image_feat_dict_train = {
    adm_id: np.mean(vecs, axis=0).astype(np.float32)
    for adm_id, vecs in image_temp_train.items()
}
image_feat_dict_val = {
    adm_id: np.mean(vecs, axis=0).astype(np.float32)
    for adm_id, vecs in image_temp_val.items()
}
image_feat_dict_test = {
    adm_id: np.mean(vecs, axis=0).astype(np.float32)
    for adm_id, vecs in image_temp_test.items()
}

if image_feat_dict_train:
    example_id_img = next(iter(image_feat_dict_train.keys()))
    image_feature_dim = image_feat_dict_train[example_id_img].shape[0]
else:
    image_feature_dim = 0

print("\nImage feature dictionary sizes:")
print(f"Train: {len(image_feat_dict_train)}")
print(f"Valid: {len(image_feat_dict_val)}")
print(f"Test:  {len(image_feat_dict_test)}")
print(f"Image feature dimension: {image_feature_dim}")


Loading core datasets...
train_df shape: (49451, 8)
val_df shape:   (16721, 8)
test_df shape:  (16293, 7)
notes_df shape: (53892, 5)

Unique admissions (by id):
Train: (8234, 2)
Valid: (2788, 2)
Test:  (2741, 1)

Notes grouped shape: (53892, 2)
Loading EHR pickle...
EHR feature dimension: 171
Loading CXR image features...
Found 87472 image feature files

Image feature dictionary sizes:
Train: 8234
Valid: 2788
Test:  2741
Image feature dimension: 1024


### 1. EHR time-series baseline

The GRU is the first true model because the EHR data is sequential. I train it separately so I can see how much signal comes from temporal structure before mixing it with the other modalities.

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy import sparse
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

SEED = 2025
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
ehr_gru_device = torch.device("cuda" if torch.cuda.is_available() else "mps" if HAS_MPS else "cpu")
print(f"Using device: {ehr_gru_device}")

EHR_DATA_DIR = "/Users/alex/30-days-hospital-readmission-rate/2025-fall-stat-3612-group-project"
EHR_TRAIN_PATH = os.path.join(EHR_DATA_DIR, "train.csv")
EHR_VAL_PATH = os.path.join(EHR_DATA_DIR, "valid.csv")
EHR_TEST_PATH = os.path.join(EHR_DATA_DIR, "test.csv")
EHR_PKL_PATH = os.path.join(EHR_DATA_DIR, "ehr_preprocessed_seq_by_day_cat_embedding.pkl")
EHR_SUBMISSION_PATH = os.path.join(EHR_DATA_DIR, "submission_ehr_gru_l2.csv")


def to_binary(series):
    return (
        series.astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        .astype(int)
    )


print("Loading EHR data...")
ehr_gru_train_df = pd.read_csv(EHR_TRAIN_PATH, usecols=["id", "readmitted_within_30days"]).drop_duplicates("id")
ehr_gru_val_df = pd.read_csv(EHR_VAL_PATH, usecols=["id", "readmitted_within_30days"]).drop_duplicates("id")
ehr_gru_test_df = pd.read_csv(EHR_TEST_PATH, usecols=["id"]).drop_duplicates("id")

ehr_gru_train_df["readmitted_within_30days"] = to_binary(ehr_gru_train_df["readmitted_within_30days"])
ehr_gru_val_df["readmitted_within_30days"] = to_binary(ehr_gru_val_df["readmitted_within_30days"])

ehr_gru_data = pd.read_pickle(EHR_PKL_PATH)
ehr_gru_feat_dict = ehr_gru_data["feat_dict"]
ehr_gru_input_dim = next(iter(ehr_gru_feat_dict.values())).shape[1]
print(f"EHR input dimension: {ehr_gru_input_dim}")


def collect_training_rows(core_df):
    rows = []
    for adm_id in core_df["id"]:
        if adm_id in ehr_gru_feat_dict:
            rows.append(np.asarray(ehr_gru_feat_dict[adm_id], dtype=np.float32))
    if not rows:
        raise ValueError("No EHR rows were found for scaler fitting.")
    return np.vstack(rows)


ehr_gru_scaler = StandardScaler()
ehr_gru_scaler.fit(collect_training_rows(ehr_gru_train_df))


class EHRDataset(Dataset):
    def __init__(self, core_df, feat_dict, scaler=None, has_label=True, input_dim=None):
        self.ids = []
        self.seqs = []
        self.lengths = []
        self.labels = []
        self.has_label = has_label
        self.input_dim = input_dim
        self.feat_dict = feat_dict
        self.scaler = scaler

        for _, row in core_df.iterrows():
            adm_id = row["id"]
            if adm_id in feat_dict:
                seq = np.asarray(feat_dict[adm_id], dtype=np.float32)
            else:
                seq = np.zeros((1, self.input_dim), dtype=np.float32)
            if scaler is not None:
                seq = scaler.transform(seq).astype(np.float32)
            self.ids.append(adm_id)
            self.seqs.append(seq)
            self.lengths.append(seq.shape[0])
            if has_label:
                self.labels.append(float(row["readmitted_within_30days"]))

        self.labels = np.asarray(self.labels, dtype=np.float32) if has_label else None
        print(f"EHRDataset(has_label={has_label}) -> {len(self.ids)} rows")

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        label = -1.0 if self.labels is None else self.labels[idx]
        return self.seqs[idx], self.lengths[idx], label, self.ids[idx]



def ehr_gru_collate(batch):
    seqs, lengths, labels, ids = zip(*batch)
    seq_tensors = [torch.tensor(seq, dtype=torch.float32) for seq in seqs]
    x = pad_sequence(seq_tensors, batch_first=True)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.float32)
    return x, lengths, labels, list(ids)


class EHRGRUExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, bidirectional=True, dropout=0.2):
        super().__init__()
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        feature_dim = hidden_dim * (2 if bidirectional else 1)
        self.feature_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feature_dim, 1)

    def encode(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        if self.bidirectional:
            features = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            features = h_n[-1]
        return features

    def forward(self, x, lengths, return_features=False):
        features = self.encode(x, lengths)
        logits = self.classifier(self.feature_dropout(features)).squeeze(1)
        if return_features:
            return logits, features
        return logits


@torch.no_grad()
def eval_gru_model(model, loader, criterion):
    model.eval()
    losses = []
    labels_all = []
    probs_all = []

    for x, lengths, labels, _ in loader:
        x = x.to(ehr_gru_device)
        lengths = lengths.to(ehr_gru_device)
        labels = labels.to(ehr_gru_device)

        logits = model(x, lengths)
        loss = criterion(logits, labels)
        probs = torch.sigmoid(logits).cpu().numpy()

        losses.append(loss.item())
        labels_all.append(labels.cpu().numpy())
        probs_all.append(probs)

    labels_all = np.concatenate(labels_all)
    probs_all = np.concatenate(probs_all)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except ValueError:
        auc = np.nan
    return float(np.mean(losses)), auc, labels_all, probs_all



def train_gru_model(model, train_loader, valid_loader, train_labels, epochs=5, lr=1e-3):
    pos_weight_value = np.float32((len(train_labels) - train_labels.sum()) / max(train_labels.sum(), 1.0))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=ehr_gru_device, dtype=torch.float32))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state = None
    best_auc = -1.0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for x, lengths, labels, _ in train_loader:
            x = x.to(ehr_gru_device)
            lengths = lengths.to(ehr_gru_device)
            labels = labels.to(ehr_gru_device)

            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_loss, val_auc, _, _ = eval_gru_model(model, valid_loader, criterion)
        print(f"Epoch {epoch:02d} | train_loss={np.mean(train_losses):.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(ehr_gru_device)
    return model, best_auc


@torch.no_grad()
def extract_gru_features(model, loader):
    model.eval()
    features = []
    ids_all = []
    for x, lengths, labels, ids in loader:
        x = x.to(ehr_gru_device)
        lengths = lengths.to(ehr_gru_device)
        _, latent = model(x, lengths, return_features=True)
        features.append(latent.cpu().numpy().astype(np.float32))
        ids_all.extend(ids)
    return np.vstack(features), ids_all


ehr_gru_train_ds = EHRDataset(
    ehr_gru_train_df,
    feat_dict=ehr_gru_feat_dict,
    scaler=ehr_gru_scaler,
    has_label=True,
    input_dim=ehr_gru_input_dim,
)
ehr_gru_val_ds = EHRDataset(
    ehr_gru_val_df,
    feat_dict=ehr_gru_feat_dict,
    scaler=ehr_gru_scaler,
    has_label=True,
    input_dim=ehr_gru_input_dim,
)
ehr_gru_test_ds = EHRDataset(
    ehr_gru_test_df,
    feat_dict=ehr_gru_feat_dict,
    scaler=ehr_gru_scaler,
    has_label=False,
    input_dim=ehr_gru_input_dim,
)

ehr_gru_train_loader = DataLoader(ehr_gru_train_ds, batch_size=64, shuffle=True, collate_fn=ehr_gru_collate)
ehr_gru_train_eval_loader = DataLoader(ehr_gru_train_ds, batch_size=64, shuffle=False, collate_fn=ehr_gru_collate)
ehr_gru_val_loader = DataLoader(ehr_gru_val_ds, batch_size=64, shuffle=False, collate_fn=ehr_gru_collate)
ehr_gru_test_loader = DataLoader(ehr_gru_test_ds, batch_size=64, shuffle=False, collate_fn=ehr_gru_collate)

ehr_gru_model = EHRGRUExtractor(ehr_gru_input_dim, hidden_dim=128, bidirectional=True, dropout=0.2).to(ehr_gru_device)
ehr_gru_model, ehr_gru_best_auc = train_gru_model(
    ehr_gru_model,
    ehr_gru_train_loader,
    ehr_gru_val_loader,
    ehr_gru_train_ds.labels,
    epochs=5,
    lr=1e-3,
)
print(f"Best GRU validation AUC: {ehr_gru_best_auc:.4f}")

ehr_gru_train_feat, ehr_gru_train_ids = extract_gru_features(ehr_gru_model, ehr_gru_train_eval_loader)
ehr_gru_val_feat, ehr_gru_val_ids = extract_gru_features(ehr_gru_model, ehr_gru_val_loader)
ehr_gru_test_feat, ehr_gru_test_ids = extract_gru_features(ehr_gru_model, ehr_gru_test_loader)

print(f"GRU feature shapes -> train: {ehr_gru_train_feat.shape}, valid: {ehr_gru_val_feat.shape}, test: {ehr_gru_test_feat.shape}")

ehr_gru_train_feat_scaler = StandardScaler()
ehr_gru_train_feat_s = ehr_gru_train_feat_scaler.fit_transform(ehr_gru_train_feat)
ehr_gru_val_feat_s = ehr_gru_train_feat_scaler.transform(ehr_gru_val_feat)
ehr_gru_test_feat_s = ehr_gru_train_feat_scaler.transform(ehr_gru_test_feat)

ehr_gru_y_train = ehr_gru_train_ds.labels
ehr_gru_y_val = ehr_gru_val_ds.labels

best_auc = -1.0
best_c = None
best_lr_model = None
for C in [0.25, 0.5, 1.0, 2.0]:
    candidate_lr = LogisticRegression(
        solver="lbfgs",
        C=C,
        class_weight="balanced",
        max_iter=4000,
        random_state=SEED,
    )
    candidate_lr.fit(ehr_gru_train_feat_s, ehr_gru_y_train)
    ehr_gru_val_probs = candidate_lr.predict_proba(ehr_gru_val_feat_s)[:, 1]
    auc = roc_auc_score(ehr_gru_y_val, ehr_gru_val_probs)
    print(f"L2 LogisticRegression C={C}: validation ROC-AUC={auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        best_c = C
        best_lr_model = candidate_lr

print(f"Best L2 LogisticRegression C={best_c}, validation AUC={best_auc:.4f}")

ehr_gru_val_probs = best_lr_model.predict_proba(ehr_gru_val_feat_s)[:, 1]
ehr_gru_val_preds = (ehr_gru_val_probs >= 0.5).astype(int)
print(f"Validation ROC-AUC: {roc_auc_score(ehr_gru_y_val, ehr_gru_val_probs):.4f}")
print("Classification report:")
print(classification_report(ehr_gru_y_val, ehr_gru_val_preds, target_names=["No Readmit", "Readmit"]))
print("Confusion matrix:")
print(confusion_matrix(ehr_gru_y_val, ehr_gru_val_preds))

# Final prediction model: refit the logistic regression on train + valid GRU features.
ehr_gru_full_feat = np.vstack([ehr_gru_train_feat, ehr_gru_val_feat])
ehr_gru_full_y = np.concatenate([ehr_gru_y_train, ehr_gru_y_val])
ehr_gru_full_feat_scaler = StandardScaler()
ehr_gru_full_feat_s = ehr_gru_full_feat_scaler.fit_transform(ehr_gru_full_feat)
ehr_gru_test_feat_final_s = ehr_gru_full_feat_scaler.transform(ehr_gru_test_feat)

final_lr_model = LogisticRegression(
    solver="lbfgs",
    C=best_c,
    class_weight="balanced",
    max_iter=4000,
    random_state=SEED,
)
final_lr_model.fit(ehr_gru_full_feat_s, ehr_gru_full_y)

ehr_gru_test_probs = final_lr_model.predict_proba(ehr_gru_test_feat_final_s)[:, 1]
ehr_gru_submission = pd.DataFrame({
    "id": ehr_gru_test_ids,
    "readmitted_within_30days": ehr_gru_test_probs,
})
ehr_gru_submission.to_csv(EHR_SUBMISSION_PATH, index=False)
print(f"Saved submission to: {EHR_SUBMISSION_PATH}")
print(ehr_gru_submission.head())


Using device: mps
Loading EHR data...
EHR input dimension: 171
EHRDataset(has_label=True) -> 8234 rows
EHRDataset(has_label=True) -> 2788 rows
EHRDataset(has_label=False) -> 2741 rows
Epoch 01 | train_loss=0.9345 | val_loss=0.9164 | val_auc=0.7709
Epoch 02 | train_loss=0.8420 | val_loss=0.9058 | val_auc=0.7760
Epoch 03 | train_loss=0.8015 | val_loss=0.9316 | val_auc=0.7645
Epoch 04 | train_loss=0.7651 | val_loss=0.9321 | val_auc=0.7751
Epoch 05 | train_loss=0.7301 | val_loss=0.9728 | val_auc=0.7601
Best GRU validation AUC: 0.7760
GRU feature shapes -> train: (8234, 256), valid: (2788, 256), test: (2741, 256)
L2 LogisticRegression C=0.25: validation ROC-AUC=0.7558
L2 LogisticRegression C=0.5: validation ROC-AUC=0.7546
L2 LogisticRegression C=1.0: validation ROC-AUC=0.7537
L2 LogisticRegression C=2.0: validation ROC-AUC=0.7534
Best L2 LogisticRegression C=0.25, validation AUC=0.7558
Validation ROC-AUC: 0.7558
Classification report:
              precision    recall  f1-score   support

 

## 2. Notes baseline: TF-IDF + Logistic Regression

The notes branch is the main textual baseline. I use a simple TF-IDF model first because it is interpretable, fast to tune, and gives a clear reference point for whether a more complex language model is actually adding value.

This was the single modality I expected to perform best before fusion.

In [3]:
import os
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 2025
np.random.seed(SEED)

DATA_DIR = "/Users/alex/30-days-hospital-readmission-rate/2025-fall-stat-3612-group-project"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VAL_PATH = os.path.join(DATA_DIR, "valid.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
NOTES_PATH = os.path.join(DATA_DIR, "notes.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission_tfidf_l2_1000.csv")


def to_binary(series):
    return (
        series.astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        .astype(int)
    )


def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r"___+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH, usecols=["id", "readmitted_within_30days"]).drop_duplicates("id")
val_df = pd.read_csv(VAL_PATH, usecols=["id", "readmitted_within_30days"]).drop_duplicates("id")
test_df = pd.read_csv(TEST_PATH, usecols=["id"]).drop_duplicates("id")
notes_df = pd.read_csv(NOTES_PATH, usecols=["id", "text"]).copy()
notes_df["text"] = notes_df["text"].fillna("").astype(str)

train_df["readmitted_within_30days"] = to_binary(train_df["readmitted_within_30days"])
val_df["readmitted_within_30days"] = to_binary(val_df["readmitted_within_30days"])

notes_grouped = notes_df.groupby("id", as_index=False)["text"].agg(" ".join)
notes_grouped["text"] = notes_grouped["text"].map(clean_text)

train_data = train_df.merge(notes_grouped, on="id", how="left")
val_data = val_df.merge(notes_grouped, on="id", how="left")
test_data = test_df.merge(notes_grouped, on="id", how="left")

for frame in (train_data, val_data, test_data):
    frame["text"] = frame["text"].fillna("").map(clean_text)

print(f"Train admissions: {len(train_data)}")
print(f"Valid admissions: {len(val_data)}")
print(f"Test admissions:  {len(test_data)}")

vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
)

X_train = vectorizer.fit_transform(train_data["text"])
X_val = vectorizer.transform(val_data["text"])
X_test = vectorizer.transform(test_data["text"])

y_train = train_data["readmitted_within_30days"].astype(int).to_numpy()
y_val = val_data["readmitted_within_30days"].astype(int).to_numpy()

print(f"TF-IDF shapes -> train: {X_train.shape}, valid: {X_val.shape}, test: {X_test.shape}")

model = LogisticRegression(
    solver="liblinear",
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED,
)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)
val_auc = roc_auc_score(y_val, val_probs)

print(f"Validation ROC-AUC: {val_auc:.4f}")
print("Classification report:")
print(classification_report(y_val, val_preds, target_names=["No Readmit", "Readmit"]))
print("Confusion matrix:")
print(confusion_matrix(y_val, val_preds))

feature_names = vectorizer.get_feature_names_out()
coef = model.coef_.ravel()
coef_df = pd.DataFrame({"feature": feature_names, "weight": coef})
print("Top positive features:")
display(coef_df.sort_values("weight", ascending=False).head(15))
print("Top negative features:")
display(coef_df.sort_values("weight", ascending=True).head(15))

# Final model: refit on train + validation, then predict test
full_data = pd.concat([train_data, val_data], ignore_index=True)
full_y = full_data["readmitted_within_30days"].astype(int).to_numpy()

final_vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
)
X_full = final_vectorizer.fit_transform(full_data["text"])
X_test_final = final_vectorizer.transform(test_data["text"])

final_model = LogisticRegression(
    solver="liblinear",
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED,
)
final_model.fit(X_full, full_y)

test_probs = final_model.predict_proba(X_test_final)[:, 1]
submission = pd.DataFrame({
    "id": test_data["id"].values,
    "readmitted_within_30days": test_probs,
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved submission to: {SUBMISSION_PATH}")
print(submission.head())


Loading data...
Train admissions: 8234
Valid admissions: 2788
Test admissions:  2741
TF-IDF shapes -> train: (8234, 1000), valid: (2788, 1000), test: (2741, 1000)
Validation ROC-AUC: 0.8319
Classification report:
              precision    recall  f1-score   support

  No Readmit       0.93      0.86      0.89      2307
     Readmit       0.51      0.68      0.58       481

    accuracy                           0.83      2788
   macro avg       0.72      0.77      0.74      2788
weighted avg       0.86      0.83      0.84      2788

Confusion matrix:
[[1986  321]
 [ 152  329]]
Top positive features:


,feature,weight
403,family,4.369692
339,disease,2.925233
464,history,2.868762
116,admission,2.330091
306,date,2.114916
716,poor,2.104405
993,worsening,2.056013
40,22,2.048888
324,died,1.877777
795,respiratory,1.832922


Top negative features:


,feature,weight
344,disposition extended,-3.043954
420,follow,-2.980896
468,home,-2.952473
219,care facility,-2.772996
114,activity status,-2.721493
393,extended care,-2.706142
263,condition mental,-2.695693
264,consciousness,-2.618148
313,dear,-2.588820
539,level consciousness,-2.588669


Saved submission to: /Users/alex/30-days-hospital-readmission-rate/2025-fall-stat-3612-group-project/submission_tfidf_l2_1000.csv
                  id  readmitted_within_30days
0  19661325_29884966                  0.222694
1  16300198_24781018                  0.136344
2  16706302_24530345                  0.900190
3  13205882_26134830                  0.437149
4  14671276_21169914                  0.307808


## 3. Image baseline: admission-level CXR features

The image branch checks whether the admission-level CXR embeddings carry standalone readmission signal. I keep this model small on purpose so the learned latent representation is compact and easy to combine later.

In [4]:
import os

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

SEED = 2025
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
image_mlp_device = torch.device("cuda" if torch.cuda.is_available() else "mps" if HAS_MPS else "cpu")
print(f"Using device: {image_mlp_device}")

required_names = [
    "modified_train_unique",
    "modified_valid_unique",
    "modified_test_unique",
    "image_feat_dict_train",
    "image_feat_dict_val",
    "image_feat_dict_test",
    "image_feature_dim",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise ValueError(f"Missing notebook variables: {missing_names}. Run the earlier image-loading cell first.")

if image_feature_dim == 0:
    raise ValueError("image_feature_dim is 0, so no image features are available.")

IMAGE_MLP_SUBMISSION_PATH = os.path.join(DATA_DIR, "submission_image_mlp_l2.csv")


def build_image_matrix(core_df, feat_dict, image_dim, has_label=True):
    vectors = []
    labels = []

    for _, row in core_df.iterrows():
        adm_id = row["id"]
        vec = feat_dict.get(adm_id)
        if vec is None:
            vec = np.zeros(image_dim, dtype=np.float32)
        vec = np.asarray(vec, dtype=np.float32).reshape(-1)
        if vec.shape[0] != image_dim:
            vec = np.zeros(image_dim, dtype=np.float32)
        vectors.append(vec)
        if has_label:
            labels.append(float(row["readmitted_within_30days"]))

    X = np.vstack(vectors).astype(np.float32)
    y = np.asarray(labels, dtype=np.float32) if has_label else None
    return X, y


X_image_train_raw, y_image_train = build_image_matrix(
    modified_train_unique,
    image_feat_dict_train,
    image_feature_dim,
    has_label=True,
)
X_image_val_raw, y_image_val = build_image_matrix(
    modified_valid_unique,
    image_feat_dict_val,
    image_feature_dim,
    has_label=True,
)
X_image_test_raw, _ = build_image_matrix(
    modified_test_unique,
    image_feat_dict_test,
    image_feature_dim,
    has_label=False,
)

print(f"Raw image shapes -> train: {X_image_train_raw.shape}, valid: {X_image_val_raw.shape}, test: {X_image_test_raw.shape}")

image_input_scaler = StandardScaler()
X_image_train = image_input_scaler.fit_transform(X_image_train_raw)
X_image_val = image_input_scaler.transform(X_image_val_raw)
X_image_test = image_input_scaler.transform(X_image_test_raw)


class ImageMLPExtractor(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, latent_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(latent_dim, 1)

    def forward(self, x, return_features=False):
        latent = self.encoder(x)
        logits = self.classifier(latent).squeeze(1)
        if return_features:
            return logits, latent
        return logits


@torch.no_grad()
def eval_image_mlp(model, loader, criterion):
    model.eval()
    losses = []
    labels_all = []
    probs_all = []

    for x, y in loader:
        x = x.to(image_mlp_device)
        y = y.to(image_mlp_device)
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits).cpu().numpy()
        losses.append(loss.item())
        labels_all.append(y.cpu().numpy())
        probs_all.append(probs)

    labels_all = np.concatenate(labels_all)
    probs_all = np.concatenate(probs_all)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except ValueError:
        auc = np.nan
    return float(np.mean(losses)), auc, labels_all, probs_all


@torch.no_grad()
def extract_image_features(model, loader):
    model.eval()
    features = []
    for batch in loader:
        x = batch[0].to(image_mlp_device)
        _, latent = model(x, return_features=True)
        features.append(latent.cpu().numpy().astype(np.float32))
    return np.vstack(features)


def make_loader(X, y=None, batch_size=256, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    if y is None:
        dataset = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.float32)
        dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


image_train_loader = make_loader(X_image_train, y_image_train, batch_size=256, shuffle=True)
image_val_loader = make_loader(X_image_val, y_image_val, batch_size=256, shuffle=False)
image_train_feat_loader = make_loader(X_image_train, batch_size=256, shuffle=False)
image_val_feat_loader = make_loader(X_image_val, batch_size=256, shuffle=False)
image_test_feat_loader = make_loader(X_image_test, batch_size=256, shuffle=False)

image_mlp_model = ImageMLPExtractor(image_feature_dim, latent_dim=64).to(image_mlp_device)
image_pos_weight = np.float32((len(y_image_train) - y_image_train.sum()) / max(y_image_train.sum(), 1.0))
image_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(image_pos_weight, device=image_mlp_device, dtype=torch.float32))
image_optimizer = torch.optim.Adam(image_mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_image_auc = -1.0
best_image_state = None
for epoch in range(1, 6):
    image_mlp_model.train()
    train_losses = []
    for x, y in image_train_loader:
        x = x.to(image_mlp_device)
        y = y.to(image_mlp_device)
        image_optimizer.zero_grad()
        logits = image_mlp_model(x)
        loss = image_criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(image_mlp_model.parameters(), 5.0)
        image_optimizer.step()
        train_losses.append(loss.item())

    val_loss, val_auc, _, _ = eval_image_mlp(image_mlp_model, image_val_loader, image_criterion)
    print(f"Epoch {epoch:02d} | train_loss={np.mean(train_losses):.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.4f}")

    if val_auc > best_image_auc:
        best_image_auc = val_auc
        best_image_state = {k: v.detach().cpu().clone() for k, v in image_mlp_model.state_dict().items()}

if best_image_state is not None:
    image_mlp_model.load_state_dict(best_image_state)
    image_mlp_model.to(image_mlp_device)

print(f"Best image MLP validation AUC: {best_image_auc:.4f}")

X_image_train_latent = extract_image_features(image_mlp_model, image_train_feat_loader)
X_image_val_latent = extract_image_features(image_mlp_model, image_val_feat_loader)
X_image_test_latent = extract_image_features(image_mlp_model, image_test_feat_loader)
print(f"Latent feature shapes -> train: {X_image_train_latent.shape}, valid: {X_image_val_latent.shape}, test: {X_image_test_latent.shape}")

image_latent_scaler = StandardScaler()
X_image_train_latent_s = image_latent_scaler.fit_transform(X_image_train_latent)
X_image_val_latent_s = image_latent_scaler.transform(X_image_val_latent)
X_image_test_latent_s = image_latent_scaler.transform(X_image_test_latent)

best_lr_auc = -1.0
best_lr_c = None
best_lr_model = None
for C in [0.25, 0.5, 1.0, 2.0]:
    candidate_lr = LogisticRegression(
        solver="lbfgs",
        C=C,
        class_weight="balanced",
        max_iter=4000,
        random_state=SEED,
    )
    candidate_lr.fit(X_image_train_latent_s, y_image_train)
    val_probs = candidate_lr.predict_proba(X_image_val_latent_s)[:, 1]
    auc = roc_auc_score(y_image_val, val_probs)
    print(f"L2 LogisticRegression C={C}: validation ROC-AUC={auc:.4f}")
    if auc > best_lr_auc:
        best_lr_auc = auc
        best_lr_c = C
        best_lr_model = candidate_lr

print(f"Best L2 LogisticRegression C={best_lr_c}, validation AUC={best_lr_auc:.4f}")

val_probs = best_lr_model.predict_proba(X_image_val_latent_s)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)
print(f"Validation ROC-AUC: {roc_auc_score(y_image_val, val_probs):.4f}")
print("Classification report:")
print(classification_report(y_image_val, val_preds, target_names=["No Readmit", "Readmit"]))
print("Confusion matrix:")
print(confusion_matrix(y_image_val, val_preds))

X_image_full_latent = np.vstack([X_image_train_latent, X_image_val_latent])
y_image_full = np.concatenate([y_image_train, y_image_val])
full_latent_scaler = StandardScaler()
X_image_full_latent_s = full_latent_scaler.fit_transform(X_image_full_latent)
X_image_test_final_s = full_latent_scaler.transform(X_image_test_latent)

final_lr_model = LogisticRegression(
    solver="lbfgs",
    C=best_lr_c,
    class_weight="balanced",
    max_iter=4000,
    random_state=SEED,
)
final_lr_model.fit(X_image_full_latent_s, y_image_full)

image_test_probs = final_lr_model.predict_proba(X_image_test_final_s)[:, 1]
image_submission = pd.DataFrame({
    "id": modified_test_unique["id"].values,
    "readmitted_within_30days": image_test_probs,
})
image_submission.to_csv(IMAGE_MLP_SUBMISSION_PATH, index=False)
print(f"Saved submission to: {IMAGE_MLP_SUBMISSION_PATH}")
print(image_submission.head())


Using device: mps
Raw image shapes -> train: (8234, 1024), valid: (2788, 1024), test: (2741, 1024)
Epoch 01 | train_loss=1.0447 | val_loss=1.0395 | val_auc=0.6915
Epoch 02 | train_loss=0.9900 | val_loss=1.0565 | val_auc=0.6927
Epoch 03 | train_loss=0.9433 | val_loss=1.0891 | val_auc=0.6835
Epoch 04 | train_loss=0.9255 | val_loss=1.0778 | val_auc=0.6865
Epoch 05 | train_loss=0.8910 | val_loss=1.0680 | val_auc=0.6978
Best image MLP validation AUC: 0.6978
Latent feature shapes -> train: (8234, 64), valid: (2788, 64), test: (2741, 64)
L2 LogisticRegression C=0.25: validation ROC-AUC=0.6830
L2 LogisticRegression C=0.5: validation ROC-AUC=0.6829
L2 LogisticRegression C=1.0: validation ROC-AUC=0.6829
L2 LogisticRegression C=2.0: validation ROC-AUC=0.6829
Best L2 LogisticRegression C=0.25, validation AUC=0.6830
Validation ROC-AUC: 0.6830
Classification report:
              precision    recall  f1-score   support

  No Readmit       0.89      0.70      0.78      2307
     Readmit       0.29   

## 4. Late fusion of EHR, notes, and image features

Once the single-modality baselines are in place, I concatenate the learned EHR, text, and image representations and train a small MLP. The idea is that each modality captures a different part of the patient story, and the interaction between them may be nonlinear.

### 5-fold cross validation for the fused MLP

This cell tunes the fusion model with stratified 5-fold CV and uses the best fold setting for the final submission. I prefer this over a single split because the stacked features are high-dimensional and the model can overfit a one-off validation split.

In [9]:
import os

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy import sparse

# 5-fold CV tuner for the stacked TF-IDF + GRU + image feature matrix.
# This cell reuses the combined labeled features from Cell 10, searches a small
# MLP hyperparameter grid with stratified 5-fold cross validation, and then
# averages the best fold models for the final submission.

def _to_binary(series):
    return (
        pd.Series(series)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
        .astype(np.int64)
        .to_numpy()
    )

required_base_names = [
    "X_full",
    "X_test_final",
    "ehr_gru_full_feat",
    "ehr_gru_test_feat",
    "X_image_full_latent",
    "X_image_test_latent",
    "modified_train_unique",
    "modified_valid_unique",
    "modified_test_unique",
]
missing_base_names = [name for name in required_base_names if name not in globals()]
if missing_base_names:
    raise ValueError(f"Missing required feature variables: {missing_base_names}")

if "X_combo_full" not in globals():
    print("Rebuilding X_combo_full from TF-IDF, GRU, and image blocks...")
    X_combo_full = sparse.hstack(
        [
            X_full.astype(np.float32),
            sparse.csr_matrix(np.asarray(ehr_gru_full_feat, dtype=np.float32)),
            sparse.csr_matrix(np.asarray(X_image_full_latent, dtype=np.float32)),
        ]
    ).tocsr()

if "X_combo_test_final" not in globals():
    print("Rebuilding X_combo_test_final from TF-IDF, GRU, and image blocks...")
    X_combo_test_final = sparse.hstack(
        [
            X_test_final.astype(np.float32),
            sparse.csr_matrix(np.asarray(ehr_gru_test_feat, dtype=np.float32)),
            sparse.csr_matrix(np.asarray(X_image_test_latent, dtype=np.float32)),
        ]
    ).tocsr()

if "y_combo_full" not in globals():
    print("Rebuilding y_combo_full from train+valid labels...")
    y_combo_full = np.concatenate([
        _to_binary(modified_train_unique["readmitted_within_30days"]),
        _to_binary(modified_valid_unique["readmitted_within_30days"]),
    ])

SEED = 2025
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
cv_device = combo_mlp_device if "combo_mlp_device" in globals() else torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if HAS_MPS else "cpu"
)
print(f"Using device: {cv_device}")

X_cv_raw = X_combo_full.toarray().astype(np.float32, copy=False)
y_cv = np.asarray(y_combo_full, dtype=np.int64)
X_test_raw = X_combo_test_final.toarray().astype(np.float32, copy=False)

print(f"CV feature shapes -> full: {X_cv_raw.shape}, test: {X_test_raw.shape}")
print(f"Positive rate in full labeled set: {y_cv.mean():.4f}")


class FusionMLPCV(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 128), dropout=0.3):
        super().__init__()
        layers = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


def make_loader_cv(X, y=None, batch_size=256, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    if y is None:
        dataset = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.float32)
        dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def predict_probs_cv(model, loader):
    model.eval()
    probs_all = []
    for batch in loader:
        x = batch[0].to(cv_device)
        logits = model(x)
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs_all)


@torch.no_grad()
def evaluate_cv_model(model, loader, criterion):
    model.eval()
    losses = []
    labels_all = []
    probs_all = []

    for x, y in loader:
        x = x.to(cv_device)
        y = y.to(cv_device)
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits).cpu().numpy()
        losses.append(loss.item())
        labels_all.append(y.cpu().numpy())
        probs_all.append(probs)

    labels_all = np.concatenate(labels_all)
    probs_all = np.concatenate(probs_all)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except ValueError:
        auc = np.nan
    return float(np.mean(losses)), auc, labels_all, probs_all


def fit_cv_fold(X_train_raw, y_train, X_val_raw, y_val, config, fold_seed):
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32, copy=False)
    X_val = scaler.transform(X_val_raw).astype(np.float32, copy=False)

    train_loader = make_loader_cv(X_train, y_train, batch_size=config["batch_size"], shuffle=True)
    val_loader = make_loader_cv(X_val, y_val, batch_size=config["batch_size"], shuffle=False)

    model = FusionMLPCV(
        input_dim=X_train.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(cv_device)

    pos_weight_value = np.float32((len(y_train) - y_train.sum()) / max(y_train.sum(), 1.0))
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight_value, device=cv_device, dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
    )

    best_state = None
    best_auc = -1.0
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        train_losses = []
        for x, y in train_loader:
            x = x.to(cv_device)
            y = y.to(cv_device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_loss, val_auc, _, _ = evaluate_cv_model(model, val_loader, criterion)
        scheduler.step(val_auc)
        print(
            f"    Epoch {epoch:02d} | train_loss={np.mean(train_losses):.4f} | "
            f"val_loss={val_loss:.4f} | val_auc={val_auc:.4f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= config["patience"]:
                print(f"    Early stopping at epoch {epoch:02d}")
                break

    if best_state is None:
        raise RuntimeError("Cross-validation fold training did not produce a best state.")

    model.load_state_dict(best_state)
    model.to(cv_device)
    val_probs = predict_probs_cv(model, val_loader)

    return {
        "scaler": scaler,
        "state_dict": best_state,
        "best_auc": best_auc,
        "best_epoch": best_epoch,
        "val_probs": val_probs,
    }


cv_configs = [
    {
        "name": "baseline_winner",
        "hidden_dims": (512, 128),
        "dropout": 0.30,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 256,
        "max_epochs": 15,
        "patience": 3,
    },
    {
        "name": "more_dropout",
        "hidden_dims": (384, 128),
        "dropout": 0.35,
        "lr": 8e-4,
        "weight_decay": 1e-4,
        "batch_size": 256,
        "max_epochs": 15,
        "patience": 3,
    },
    {
        "name": "wider_shallower",
        "hidden_dims": (512, 256),
        "dropout": 0.25,
        "lr": 8e-4,
        "weight_decay": 5e-5,
        "batch_size": 256,
        "max_epochs": 15,
        "patience": 3,
    },
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_summary_rows = []
best_cv_mean_auc = -1.0
best_cv_oof_auc = -1.0
best_cv_config = None
best_cv_fold_artifacts = None

for config in cv_configs:
    print(
        f"\nConfig: {config['name']} | hidden_dims={config['hidden_dims']} | "
        f"dropout={config['dropout']} | lr={config['lr']} | weight_decay={config['weight_decay']}"
    )
    oof_probs = np.zeros(len(y_cv), dtype=np.float32)
    fold_scores = []
    fold_epochs = []
    fold_artifacts = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_cv_raw, y_cv), start=1):
        print(f"  Fold {fold_idx}/5")
        fold_result = fit_cv_fold(
            X_cv_raw[train_idx],
            y_cv[train_idx],
            X_cv_raw[val_idx],
            y_cv[val_idx],
            config,
            fold_seed=SEED + fold_idx,
        )
        fold_scores.append(fold_result["best_auc"])
        fold_epochs.append(fold_result["best_epoch"])
        fold_artifacts.append(fold_result)
        oof_probs[val_idx] = fold_result["val_probs"]
        print(f"  Fold {fold_idx} best AUC={fold_result['best_auc']:.4f} at epoch {fold_result['best_epoch']}")

    mean_fold_auc = float(np.mean(fold_scores))
    std_fold_auc = float(np.std(fold_scores))
    oof_auc = roc_auc_score(y_cv, oof_probs)
    oof_preds = (oof_probs >= 0.5).astype(int)

    print(f"Config mean fold AUC: {mean_fold_auc:.4f} ± {std_fold_auc:.4f}")
    print(f"Config OOF AUC: {oof_auc:.4f}")
    print("OOF classification report:")
    print(classification_report(y_cv, oof_preds, target_names=["No Readmit", "Readmit"]))
    print("OOF confusion matrix:")
    print(confusion_matrix(y_cv, oof_preds))

    cv_summary_rows.append({
        "config": config["name"],
        "hidden_dims": str(config["hidden_dims"]),
        "dropout": config["dropout"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "mean_fold_auc": mean_fold_auc,
        "std_fold_auc": std_fold_auc,
        "oof_auc": oof_auc,
        "mean_best_epoch": float(np.mean(fold_epochs)),
    })

    if (mean_fold_auc > best_cv_mean_auc) or (
        np.isclose(mean_fold_auc, best_cv_mean_auc) and oof_auc > best_cv_oof_auc
    ):
        best_cv_mean_auc = mean_fold_auc
        best_cv_oof_auc = oof_auc
        best_cv_config = config
        best_cv_fold_artifacts = fold_artifacts

cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values(["mean_fold_auc", "oof_auc"], ascending=False)
print("\nCross-validation summary:")
display(cv_summary_df)
print(f"Best CV config: {best_cv_config}")
print(f"Best mean fold AUC: {best_cv_mean_auc:.4f}")
print(f"Best OOF AUC: {best_cv_oof_auc:.4f}")

# Average the best config's 5 fold models for the final test predictions.
assert best_cv_fold_artifacts is not None

test_prob_sum = np.zeros(X_test_raw.shape[0], dtype=np.float32)
for fold_idx, fold_artifact in enumerate(best_cv_fold_artifacts, start=1):
    fold_scaler = fold_artifact["scaler"]
    X_test_fold = fold_scaler.transform(X_test_raw).astype(np.float32, copy=False)
    test_loader = make_loader_cv(X_test_fold, batch_size=best_cv_config["batch_size"], shuffle=False)
    fold_model = FusionMLPCV(
        input_dim=X_test_fold.shape[1],
        hidden_dims=best_cv_config["hidden_dims"],
        dropout=best_cv_config["dropout"],
    ).to(cv_device)
    fold_model.load_state_dict(fold_artifact["state_dict"])
    fold_model.to(cv_device)
    fold_test_probs = predict_probs_cv(fold_model, test_loader)
    test_prob_sum += fold_test_probs.astype(np.float32)
    print(f"Test ensemble fold {fold_idx}/5 done")

test_probs = test_prob_sum / len(best_cv_fold_artifacts)
mlp_cv_submission = pd.DataFrame({
    "id": modified_test_unique["id"].values,
    "readmitted_within_30days": test_probs,
})
MLP_CV_SUBMISSION_PATH = os.path.join(DATA_DIR, "submission_mlp_5fold_cv.csv")
mlp_cv_submission.to_csv(MLP_CV_SUBMISSION_PATH, index=False)
print(f"Saved submission to: {MLP_CV_SUBMISSION_PATH}")
print(mlp_cv_submission.head())


Using device: mps
CV feature shapes -> full: (11022, 1320), test: (2741, 1320)
Positive rate in full labeled set: 0.1751

Config: baseline_winner | hidden_dims=(512, 128) | dropout=0.3 | lr=0.001 | weight_decay=0.0001
  Fold 1/5
    Epoch 01 | train_loss=0.7235 | val_loss=0.6142 | val_auc=0.9025
    Epoch 02 | train_loss=0.5049 | val_loss=0.6576 | val_auc=0.8974
    Epoch 03 | train_loss=0.3868 | val_loss=0.7025 | val_auc=0.9000
    Epoch 04 | train_loss=0.2331 | val_loss=0.8251 | val_auc=0.8988
    Early stopping at epoch 04
  Fold 1 best AUC=0.9025 at epoch 1
  Fold 2/5
    Epoch 01 | train_loss=0.7257 | val_loss=0.6190 | val_auc=0.8997
    Epoch 02 | train_loss=0.5088 | val_loss=0.6879 | val_auc=0.8974
    Epoch 03 | train_loss=0.3713 | val_loss=0.7305 | val_auc=0.8819
    Epoch 04 | train_loss=0.2453 | val_loss=0.9173 | val_auc=0.8817
    Early stopping at epoch 04
  Fold 2 best AUC=0.8997 at epoch 1
  Fold 3/5
    Epoch 01 | train_loss=0.7110 | val_loss=0.6577 | val_auc=0.8849
   

,config,hidden_dims,dropout,lr,weight_decay,mean_fold_auc,std_fold_auc,oof_auc,mean_best_epoch
0,baseline_winner,"(512, 128)",0.30,0.0010,0.00010,0.894487,0.007602,0.890165,1.2
1,more_dropout,"(384, 128)",0.35,0.0008,0.00010,0.894090,0.009069,0.889259,2.0
2,wider_shallower,"(512, 256)",0.25,0.0008,0.00005,0.892239,0.007990,0.888020,1.6


Best CV config: {'name': 'baseline_winner', 'hidden_dims': (512, 128), 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 256, 'max_epochs': 15, 'patience': 3}
Best mean fold AUC: 0.8945
Best OOF AUC: 0.8902
Test ensemble fold 1/5 done
Test ensemble fold 2/5 done
Test ensemble fold 3/5 done
Test ensemble fold 4/5 done
Test ensemble fold 5/5 done
Saved submission to: /Users/alex/30-days-hospital-readmission-rate/2025-fall-stat-3612-group-project/submission_mlp_5fold_cv.csv
                  id  readmitted_within_30days
0  19661325_29884966                  0.262479
1  16300198_24781018                  0.122787
2  16706302_24530345                  0.982280
3  13205882_26134830                  0.584224
4  14671276_21169914                  0.485712


## 5. Note section screening

Before trying ClinicalBERT, I score each note section separately with TF-IDF + logistic regression. This is a cheap diagnostic: if a section already carries signal on its own, it is more likely worth feeding into a heavier encoder, and it tells me how to spend the 512-token budget.

In [11]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Build the screening table directly from the raw notes and the available train/valid splits.
# This makes the cell self-contained even if a precomputed full_notes_df is not in memory.

required_names = ["notes_df", "modified_train_unique", "modified_valid_unique"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise ValueError(f"Missing required notebook variables: {missing_names}")

SEED = 2025
N_SPLITS = 5
TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.98
LOGREG_C = 1.0
LOGREG_MAX_ITER = 2000


def to_binary(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
        .astype(np.int64)
        .to_numpy()
    )


full_core_df = pd.concat(
    [
        modified_train_unique[["id", "readmitted_within_30days"]],
        modified_valid_unique[["id", "readmitted_within_30days"]],
    ],
    ignore_index=True,
)
full_core_df["readmitted_within_30days"] = to_binary(full_core_df["readmitted_within_30days"])

notes_grouped = notes_df.copy()
notes_grouped["text"] = notes_grouped["text"].fillna("").astype(str)
notes_grouped = notes_grouped.groupby("id", as_index=False)["text"].agg("\n".join)

full_notes_df = full_core_df.merge(notes_grouped, on="id", how="left")
full_notes_df["text"] = full_notes_df["text"].fillna("").astype(str)

CANDIDATE_SECTIONS = [
    "Chief Complaint",
    "History of Present Illness",
    "Past Medical History",
    "Social History",
    "Family History",
    "Physical Exam",
    "Pertinent Results",
    "Brief Hospital Course",
    "Medications on Admission",
    "Discharge Medications",
    "Discharge Diagnosis",
    "Discharge Condition",
    "Discharge Disposition",
    "Discharge Instructions",
    "Followup Instructions",
]

SECTION_ALIASES = {
    "chief complaint": "Chief Complaint",
    "history of present illness": "History of Present Illness",
    "hpi": "History of Present Illness",
    "past medical history": "Past Medical History",
    "social history": "Social History",
    "family history": "Family History",
    "physical exam": "Physical Exam",
    "pertinent results": "Pertinent Results",
    "brief hospital course": "Brief Hospital Course",
    "medications on admission": "Medications on Admission",
    "discharge medications": "Discharge Medications",
    "medications on discharge": "Discharge Medications",
    "discharge diagnosis": "Discharge Diagnosis",
    "discharge condition": "Discharge Condition",
    "condition on discharge": "Discharge Condition",
    "discharge disposition": "Discharge Disposition",
    "discharge instructions": "Discharge Instructions",
    "followup instructions": "Followup Instructions",
}

ALL_HEADER_PATTERNS = list(dict.fromkeys(CANDIDATE_SECTIONS + ["HPI", "Medications on Discharge", "Condition on Discharge"]))
HEADER_REGEX = re.compile(
    r"(?<!\n)\s+(" + "|".join(re.escape(header) for header in ALL_HEADER_PATTERNS) + r")\s*:",
    flags=re.IGNORECASE,
)


def normalize_header(header):
    header = re.sub(r"\s+", " ", str(header).strip()).lower()
    header = re.sub(r"[\s:.\-]+$", "", header)
    return SECTION_ALIASES.get(header)



def preprocess_inline_headers(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\r\n?", "\n", text)
    return HEADER_REGEX.sub(r"\n\1:", text)



def extract_candidate_sections(text):
    text = preprocess_inline_headers(text)
    sections = {section: [] for section in CANDIDATE_SECTIONS}
    current_section = None

    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line:
            continue

        header_only = normalize_header(line)
        if header_only is not None:
            current_section = header_only
            continue

        if ":" in line:
            header_prefix, remainder = line.split(":", 1)
            mapped = normalize_header(header_prefix)
            if mapped is not None:
                current_section = mapped
                remainder = remainder.strip()
                if remainder:
                    sections[current_section].append(remainder)
                continue
            if len(header_prefix.strip()) <= 80 and re.fullmatch(r"[A-Za-z0-9 /\-&()]+", header_prefix.strip()):
                current_section = None
                continue

        if current_section is not None:
            sections[current_section].append(line)

    return {
        section: re.sub(r"\s+", " ", " ".join(sections[section]).strip())
        for section in CANDIDATE_SECTIONS
    }


section_frame = pd.DataFrame(full_notes_df["text"].map(extract_candidate_sections).tolist())
full_notes_df = pd.concat([full_notes_df, section_frame], axis=1)
for sec in CANDIDATE_SECTIONS:
    full_notes_df[sec] = full_notes_df[sec].fillna("").astype(str)

CANDIDATE_SECTIONS = [sec for sec in CANDIDATE_SECTIONS if sec in full_notes_df.columns]
print("Sections to evaluate:", CANDIDATE_SECTIONS)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
y = full_notes_df["readmitted_within_30days"].to_numpy(dtype=np.int64)


def safe_auc(y_true, y_score):
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan



def clean_section_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("___", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text



def section_summary_stats(texts):
    texts = texts.fillna("").astype(str)
    char_len = texts.str.len()
    word_len = texts.str.split().map(len)
    non_empty_rate = (char_len > 0).mean()
    return {
        "non_empty_rate": float(non_empty_rate),
        "mean_char_len": float(char_len.mean()),
        "median_char_len": float(char_len.median()),
        "mean_word_len": float(word_len.mean()),
        "median_word_len": float(word_len.median()),
    }


results = []
print("\nRunning robust section-by-section TF-IDF + Logistic Regression CV...")

for sec in CANDIDATE_SECTIONS:
    print(f"\nEvaluating section: {sec}")

    texts = full_notes_df[sec].fillna("").astype(str).map(clean_section_text)
    stats = section_summary_stats(texts)
    unique_nonempty = texts[texts.str.len() > 0].nunique()
    total_nonempty = int((texts.str.len() > 0).sum())

    print(f"  Non-empty notes: {total_nonempty}/{len(texts)} ({stats['non_empty_rate']:.2%})")
    print(f"  Unique non-empty texts: {unique_nonempty}")

    if total_nonempty < 50 or unique_nonempty < 10:
        print("  Skipping: section too sparse / too few unique values")
        results.append({
            "section": sec,
            **stats,
            "mean_fold_auc": np.nan,
            "std_fold_auc": np.nan,
            "oof_auc": np.nan,
            "status": "skipped_sparse",
        })
        continue

    fold_aucs = []
    oof_probs = np.full(len(full_notes_df), np.nan, dtype=np.float32)
    fold_failed = False

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(texts, y), start=1):
        X_train_text = texts.iloc[train_idx].tolist()
        X_val_text = texts.iloc[val_idx].tolist()
        y_train = y[train_idx]
        y_val = y[val_idx]

        train_nonempty = sum(len(t) > 0 for t in X_train_text)
        train_unique_nonempty = len(set(t for t in X_train_text if len(t) > 0))

        if train_nonempty < 20 or train_unique_nonempty < 5:
            print(f"  Fold {fold_idx}: skipped (too sparse in training fold)")
            fold_failed = True
            break

        try:
            vectorizer = TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=TFIDF_NGRAM_RANGE,
                min_df=MIN_DF,
                max_df=MAX_DF,
                lowercase=True,
                strip_accents="unicode",
                sublinear_tf=True,
            )

            X_train = vectorizer.fit_transform(X_train_text)
            X_val = vectorizer.transform(X_val_text)

            if X_train.shape[1] == 0:
                print(f"  Fold {fold_idx}: skipped (empty TF-IDF vocabulary)")
                fold_failed = True
                break

            clf = LogisticRegression(
                C=LOGREG_C,
                max_iter=LOGREG_MAX_ITER,
                class_weight="balanced",
                solver="liblinear",
                random_state=SEED,
            )

            clf.fit(X_train, y_train)
            val_probs = clf.predict_proba(X_val)[:, 1]
            fold_auc = safe_auc(y_val, val_probs)

            fold_aucs.append(fold_auc)
            oof_probs[val_idx] = val_probs
            print(f"  Fold {fold_idx}/{N_SPLITS} AUC = {fold_auc:.4f}")

        except ValueError as e:
            print(f"  Fold {fold_idx}: failed with ValueError -> {e}")
            fold_failed = True
            break

    if fold_failed:
        results.append({
            "section": sec,
            **stats,
            "mean_fold_auc": np.nan,
            "std_fold_auc": np.nan,
            "oof_auc": np.nan,
            "status": "failed_sparse_or_empty_vocab",
        })
    else:
        valid_mask = ~np.isnan(oof_probs)
        oof_auc = safe_auc(y[valid_mask], oof_probs[valid_mask])
        results.append({
            "section": sec,
            **stats,
            "mean_fold_auc": float(np.nanmean(fold_aucs)),
            "std_fold_auc": float(np.nanstd(fold_aucs)),
            "oof_auc": float(oof_auc),
            "status": "ok",
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=["oof_auc", "mean_fold_auc"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

print("\n==================== Ranked Section Results ====================")
print(results_df.to_string(index=False))

SECTION_SCREEN_RESULTS_PATH = "./section_screen_tfidf_logreg_results_robust.csv"
results_df.to_csv(SECTION_SCREEN_RESULTS_PATH, index=False)
print(f"\nSaved results to: {SECTION_SCREEN_RESULTS_PATH}")

print("\n==================== Successful Sections Only ====================")
print(results_df[results_df["status"] == "ok"].to_string(index=False))


Sections to evaluate: ['Chief Complaint', 'History of Present Illness', 'Past Medical History', 'Social History', 'Family History', 'Physical Exam', 'Pertinent Results', 'Brief Hospital Course', 'Medications on Admission', 'Discharge Medications', 'Discharge Diagnosis', 'Discharge Condition', 'Discharge Disposition', 'Discharge Instructions', 'Followup Instructions']

Running robust section-by-section TF-IDF + Logistic Regression CV...

Evaluating section: Chief Complaint
  Non-empty notes: 10559/11022 (95.80%)
  Unique non-empty texts: 5963
  Fold 1/5 AUC = 0.5919
  Fold 2/5 AUC = 0.5668
  Fold 3/5 AUC = 0.5870
  Fold 4/5 AUC = 0.6154
  Fold 5/5 AUC = 0.6276

Evaluating section: History of Present Illness
  Non-empty notes: 10914/11022 (99.02%)
  Unique non-empty texts: 10909
  Fold 1/5 AUC = 0.6424
  Fold 2/5 AUC = 0.6319
  Fold 3/5 AUC = 0.6272
  Fold 4/5 AUC = 0.6310
  Fold 5/5 AUC = 0.6307

Evaluating section: Past Medical History
  Non-empty notes: 9675/11022 (87.78%)
  Unique no

## 6. ClinicalBERT fusion on the strongest note sections

The section scan points to discharge-oriented text as the most useful note signal. I therefore feed Discharge Medications, Discharge Instructions, and Discharge Condition into Bio_ClinicalBERT, then stack those embeddings with the EHR and image latents. This is the main test of whether a pretrained medical language model can improve on the simpler TF-IDF baseline.

In [12]:
import os
import re

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# ClinicalBERT + GRU + image MLP fusion using the already extracted modality features.
# Notes are restricted to Discharge Medications, Discharge Instructions, and Discharge Condition.

required_names = [
    "notes_df",
    "modified_train_unique",
    "modified_valid_unique",
    "modified_test_unique",
    "ehr_gru_train_feat",
    "ehr_gru_val_feat",
    "ehr_gru_test_feat",
    "X_image_train_latent",
    "X_image_val_latent",
    "X_image_test_latent",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise ValueError(f"Missing required notebook variables: {missing_names}")

if "DATA_DIR" not in globals():
    DATA_DIR = "."

SEED = 2025
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if HAS_MPS else "cpu")
print(f"Using device: {DEVICE}")


def to_binary(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
        .astype(int)
    )


full_core_df = pd.concat(
    [
        modified_train_unique[["id", "readmitted_within_30days"]],
        modified_valid_unique[["id", "readmitted_within_30days"]],
    ],
    ignore_index=True,
)
full_core_df["readmitted_within_30days"] = to_binary(full_core_df["readmitted_within_30days"])

test_core_df = modified_test_unique[["id"]].copy()
y_full = full_core_df["readmitted_within_30days"].to_numpy(dtype=np.int64)

X_ehr_full_raw = np.vstack(
    [
        np.asarray(ehr_gru_train_feat, dtype=np.float32),
        np.asarray(ehr_gru_val_feat, dtype=np.float32),
    ]
).astype(np.float32, copy=False)
X_ehr_test_raw = np.asarray(ehr_gru_test_feat, dtype=np.float32)

X_img_full_raw = np.vstack(
    [
        np.asarray(X_image_train_latent, dtype=np.float32),
        np.asarray(X_image_val_latent, dtype=np.float32),
    ]
).astype(np.float32, copy=False)
X_img_test_raw = np.asarray(X_image_test_latent, dtype=np.float32)

if len(X_ehr_full_raw) != len(full_core_df):
    raise ValueError(f"EHR full features mismatch: {len(X_ehr_full_raw)} vs {len(full_core_df)}")
if len(X_img_full_raw) != len(full_core_df):
    raise ValueError(f"Image full features mismatch: {len(X_img_full_raw)} vs {len(full_core_df)}")
if len(X_ehr_test_raw) != len(modified_test_unique):
    raise ValueError(f"EHR test features mismatch: {len(X_ehr_test_raw)} vs {len(modified_test_unique)}")
if len(X_img_test_raw) != len(modified_test_unique):
    raise ValueError(f"Image test features mismatch: {len(X_img_test_raw)} vs {len(modified_test_unique)}")

print(f"EHR raw shapes -> full: {X_ehr_full_raw.shape}, test: {X_ehr_test_raw.shape}")
print(f"Image raw shapes -> full: {X_img_full_raw.shape}, test: {X_img_test_raw.shape}")
print(f"Labeled rows: {len(full_core_df)} | Positive rate: {y_full.mean():.4f}")

notes_grouped = notes_df.copy()
notes_grouped["text"] = notes_grouped["text"].fillna("").astype(str)
notes_grouped = notes_grouped.groupby("id", as_index=False)["text"].agg("\n".join)

full_notes_df = full_core_df.merge(notes_grouped, on="id", how="left")
test_notes_df = test_core_df.merge(notes_grouped, on="id", how="left")
full_notes_df["text"] = full_notes_df["text"].fillna("").astype(str)
test_notes_df["text"] = test_notes_df["text"].fillna("").astype(str)

TARGET_SECTIONS = [
    "Discharge Medications",
    "Discharge Instructions",
    "Discharge Condition",
]
SECTION_ALIASES = {
    "discharge medications": "Discharge Medications",
    "medications on discharge": "Discharge Medications",
    "discharge instructions": "Discharge Instructions",
    "discharge condition": "Discharge Condition",
    "condition on discharge": "Discharge Condition",
    "discharge meds": "Discharge Medications",
}
ALL_HEADER_PATTERNS = [
    "Discharge Medications",
    "Medications on Discharge",
    "Discharge Instructions",
    "Discharge Condition",
    "Condition on Discharge",
    "Discharge Meds",
]
HEADER_REGEX = re.compile(
    r"(?<!\n)\s+(" + "|".join(re.escape(header) for header in ALL_HEADER_PATTERNS) + r")\s*:",
    flags=re.IGNORECASE,
)


def normalize_header(header):
    header = re.sub(r"\s+", " ", str(header).strip()).lower()
    header = re.sub(r"[\s:.\-]+$", "", header)
    return SECTION_ALIASES.get(header)



def preprocess_inline_headers(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\r\n?", "\n", text)
    text = HEADER_REGEX.sub(r"\n\1:", text)
    return text



def extract_target_sections(text):
    text = preprocess_inline_headers(text)
    sections = {section: [] for section in TARGET_SECTIONS}
    current_section = None

    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line:
            continue

        header_only = normalize_header(line)
        if header_only is not None:
            current_section = header_only
            continue

        if ":" in line:
            header_prefix, remainder = line.split(":", 1)
            mapped = normalize_header(header_prefix)
            if mapped is not None:
                current_section = mapped
                remainder = remainder.strip()
                if remainder:
                    sections[current_section].append(remainder)
                continue
            if len(header_prefix.strip()) <= 80 and re.fullmatch(r"[A-Za-z0-9 /\-&()]+", header_prefix.strip()):
                current_section = None
                continue

        if current_section is not None:
            sections[current_section].append(line)

    pieces = []
    for section in TARGET_SECTIONS:
        content = re.sub(r"\s+", " ", " ".join(sections[section]).strip())
        if content:
            pieces.append(f"{section}: {content}")
    return " ".join(pieces).strip()


full_notes_df["clinical_text"] = full_notes_df["text"].apply(extract_target_sections).fillna("")
test_notes_df["clinical_text"] = test_notes_df["text"].apply(extract_target_sections).fillna("")

print(
    f"Clinical note sections found -> train+valid: {full_notes_df['clinical_text'].str.len().gt(0).sum()}/{len(full_notes_df)}, "
    f"test: {test_notes_df['clinical_text'].str.len().gt(0).sum()}/{len(test_notes_df)}"
)

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN = 512
ENCODER_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 4 if DEVICE.type == "mps" else 2

print("Loading Bio_ClinicalBERT tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
clinical_bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
clinical_bert.eval()


@torch.inference_mode()
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


@torch.inference_mode()
def encode_texts(texts, batch_size=ENCODER_BATCH_SIZE):
    texts = list(texts)
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        outputs = clinical_bert(**encoded)
        pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
        embeddings.append(pooled.cpu().numpy().astype(np.float32))
    return np.vstack(embeddings)


print("Encoding ClinicalBERT note features...")
X_note_full_raw = encode_texts(full_notes_df["clinical_text"].tolist())
X_note_test_raw = encode_texts(test_notes_df["clinical_text"].tolist())
print(f"ClinicalBERT feature shapes -> full: {X_note_full_raw.shape}, test: {X_note_test_raw.shape}")

# Free encoder memory before training the MLP.
del clinical_bert
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
elif DEVICE.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
    torch.mps.empty_cache()

X_stack_full_raw = np.hstack([X_ehr_full_raw, X_note_full_raw, X_img_full_raw]).astype(np.float32, copy=False)
X_stack_test_raw = np.hstack([X_ehr_test_raw, X_note_test_raw, X_img_test_raw]).astype(np.float32, copy=False)

print(f"Stacked feature shapes -> full: {X_stack_full_raw.shape}, test: {X_stack_test_raw.shape}")


class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 128), dropout=0.30):
        super().__init__()
        layers = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)



def make_loader(X, y=None, batch_size=256, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    if y is None:
        dataset = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.float32)
        dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


@torch.inference_mode()
def predict_probs(model, loader):
    model.eval()
    probs_all = []
    for batch in loader:
        x = batch[0].to(DEVICE)
        logits = model(x)
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs_all)


@torch.inference_mode()
def evaluate_model(model, loader, criterion):
    model.eval()
    losses = []
    labels_all = []
    probs_all = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits).cpu().numpy()
        losses.append(loss.item())
        labels_all.append(y.cpu().numpy())
        probs_all.append(probs)

    labels_all = np.concatenate(labels_all)
    probs_all = np.concatenate(probs_all)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except ValueError:
        auc = np.nan
    return float(np.mean(losses)), auc, labels_all, probs_all



def fit_fold(X_train_raw, y_train, X_val_raw, y_val, config):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32, copy=False)
    X_val = scaler.transform(X_val_raw).astype(np.float32, copy=False)

    train_loader = make_loader(X_train, y_train, batch_size=config["batch_size"], shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=config["batch_size"], shuffle=False)

    model = FusionMLP(
        input_dim=X_train.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(DEVICE)

    pos_weight_value = np.float32((len(y_train) - y_train.sum()) / max(y_train.sum(), 1.0))
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight_value, device=DEVICE, dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
    )

    best_state = None
    best_auc = -1.0
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_loss, val_auc, _, _ = evaluate_model(model, val_loader, criterion)
        scheduler.step(val_auc)
        print(
            f"    Epoch {epoch:02d} | train_loss={np.mean(train_losses):.4f} | "
            f"val_loss={val_loss:.4f} | val_auc={val_auc:.4f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= config["patience"]:
                print(f"    Early stopping at epoch {epoch:02d}")
                break

    if best_state is None:
        raise RuntimeError("No best state was saved during fold training.")

    model.load_state_dict(best_state)
    model.to(DEVICE)
    val_probs = predict_probs(model, val_loader)

    return {
        "scaler": scaler,
        "state_dict": best_state,
        "best_auc": best_auc,
        "best_epoch": best_epoch,
        "val_probs": val_probs,
    }


fusion_config = {
    "name": "gru_clinicalbert_discharge3_image_fusion",
    "hidden_dims": (512, 128),
    "dropout": 0.30,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 256,
    "max_epochs": 12,
    "patience": 3,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probs = np.zeros(len(y_full), dtype=np.float32)
fold_scores = []
fold_epochs = []
fold_artifacts = []

print(f"\nStarting 5-fold CV for fusion model: {fusion_config['name']}")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_stack_full_raw, y_full), start=1):
    print(f"\n================ Fold {fold_idx}/5 ================")
    y_train = y_full[train_idx]
    y_val = y_full[val_idx]

    X_train_fold = X_stack_full_raw[train_idx]
    X_val_fold = X_stack_full_raw[val_idx]

    fold_result = fit_fold(
        X_train_fold,
        y_train,
        X_val_fold,
        y_val,
        fusion_config,
    )

    fold_scores.append(fold_result["best_auc"])
    fold_epochs.append(fold_result["best_epoch"])
    fold_artifacts.append(fold_result)
    oof_probs[val_idx] = fold_result["val_probs"]
    print(f"  Fold {fold_idx} best AUC={fold_result['best_auc']:.4f} at epoch {fold_result['best_epoch']}")

mean_fold_auc = float(np.mean(fold_scores))
std_fold_auc = float(np.std(fold_scores))
oof_auc = roc_auc_score(y_full, oof_probs)
oof_preds = (oof_probs >= 0.5).astype(int)

print(f"\nMean fold AUC: {mean_fold_auc:.4f} ± {std_fold_auc:.4f}")
print(f"OOF AUC: {oof_auc:.4f}")
print("OOF classification report:")
print(classification_report(y_full, oof_preds, target_names=["No Readmit", "Readmit"]))
print("OOF confusion matrix:")
print(confusion_matrix(y_full, oof_preds))

# Average the fold models for test-time inference.
test_prob_sum = np.zeros(len(modified_test_unique), dtype=np.float32)
for fold_idx, fold_artifact in enumerate(fold_artifacts, start=1):
    fold_scaler = fold_artifact["scaler"]
    X_test_fold = fold_scaler.transform(X_stack_test_raw).astype(np.float32, copy=False)
    test_loader = make_loader(X_test_fold, batch_size=fusion_config["batch_size"], shuffle=False)
    fold_model = FusionMLP(
        input_dim=X_test_fold.shape[1],
        hidden_dims=fusion_config["hidden_dims"],
        dropout=fusion_config["dropout"],
    ).to(DEVICE)
    fold_model.load_state_dict(fold_artifact["state_dict"])
    fold_model.to(DEVICE)
    fold_test_probs = predict_probs(fold_model, test_loader)
    test_prob_sum += fold_test_probs.astype(np.float32)
    print(f"Test ensemble fold {fold_idx}/5 done")

test_probs = test_prob_sum / len(fold_artifacts)
submission_df = pd.DataFrame({
    "id": modified_test_unique["id"].values,
    "readmitted_within_30days": test_probs,
})
submission_path = os.path.join(
    DATA_DIR,
    "submission_gru_clinicalbert_discharge3_image_mlp_5fold_cv.csv",
)
submission_df.to_csv(submission_path, index=False)
print(f"Saved submission to: {submission_path}")
print(submission_df.head())


/Users/alex/30-days-hospital-readmission-rate/myvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
EHR raw shapes -> full: (11022, 256), test: (2741, 256)
Image raw shapes -> full: (11022, 64), test: (2741, 64)
Labeled rows: 11022 | Positive rate: 0.1751
Clinical note sections found -> train+valid: 11005/11022, test: 2738/2741
Loading Bio_ClinicalBERT tokenizer/model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 44627.41it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding ClinicalBERT note features...
ClinicalBERT feature shapes -> full: (11022, 768), test: (2741, 768)
Stacked feature shapes -> full: (11022, 1088), test: (2741, 1088)

Starting 5-fold CV for fusion model: gru_clinicalbert_discharge3_image_fusion

================ Fold 1/5 ================
    Epoch 01 | train_loss=0.6825 | val_loss=0.6148 | val_auc=0.8928
    Epoch 02 | train_loss=0.5749 | val_loss=0.6073 | val_auc=0.8967
    Epoch 03 | train_loss=0.5396 | val_loss=0.6034 | val_auc=0.8972
    Epoch 04 | train_loss=0.5204 | val_loss=0.6076 | val_auc=0.8972
    Epoch 05 | train_loss=0.4599 | val_loss=0.6415 | val_auc=0.9003
    Epoch 06 | train_loss=0.4239 | val_loss=0.6717 | val_auc=0.8923
    Epoch 07 | train_loss=0.3851 | val_loss=0.7236 | val_auc=0.8846
    Epoch 08 | train_loss=0.2942 | val_loss=0.8137 | val_auc=0.8814
    Early stopping at epoch 08
  Fold 1 best AUC=0.9003 at epoch 5

================ Fold 2/5 ================
    Epoch 01 | train_loss=0.6788 | val_loss=0.64

## Final conclusion

The notebook supports the original working hypothesis. Notes are the strongest standalone modality, image features and EHR sequences are weaker on their own, and the late-fusion MLP gives the best overall result because it can learn nonlinear interactions among modalities instead of forcing a single linear score.

ClinicalBERT did not clearly beat the simpler TF-IDF note model here for a few likely reasons. First, the 512-token limit forces the note text down to only three sections, so some relevant context is lost. Second, the implementation uses frozen mean-pooled embeddings from a pretrained encoder instead of task-specific fine-tuning, so the model cannot adapt deeply to this dataset. Third, the section-level signal in discharge notes is already strong and is captured well by TF-IDF, which makes the margin for improvement smaller.

For the Kaggle submission, I would keep submission_mlp_5fold_cv.csv, since it is the best validated model in the workbook.